# SoilVision — EDA: Crop Recommendation Dataset
**Source:** Kaggle — [Crop Recommendation Dataset by Atharva Ingle](https://www.kaggle.com/datasets/atharvaingle/crop-recommendation-dataset)  
**Purpose:** Understand feature distributions, check for class imbalance, and confirm data quality before training.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

DATA_PATH = os.path.join('data', 'Crop_recommendation.csv')
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

## 1. Basic Information

In [ ]:
print('=== Dataset Info ===')
print(df.info())
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Duplicate Rows ===')
print(df.duplicated().sum())

In [ ]:
print('=== Descriptive Statistics ===')
df.describe().round(2)

## 2. Class Distribution (22 Crops)

In [ ]:
crop_counts = df['label'].value_counts()
print(f'Number of unique crops: {len(crop_counts)}')
print(f'Rows per crop (each should be 100): ')
print(crop_counts)

fig, ax = plt.subplots(figsize=(14, 5))
crop_counts.plot(kind='bar', ax=ax, color=sns.color_palette('husl', len(crop_counts)))
ax.set_title('Crop Class Distribution — 22 Crops × 100 Samples Each', fontsize=14)
ax.set_xlabel('Crop')
ax.set_ylabel('Count')
ax.axhline(y=100, color='red', linestyle='--', label='Expected: 100')
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
print('✅ Perfect balance — 100 samples per crop. No oversampling needed.')

## 3. Feature Distributions

In [ ]:
features = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
units = ['kg/ha', 'kg/ha', 'kg/ha', '°C', '%', 'dimensionless', 'mm']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, (feat, unit) in enumerate(zip(features, units)):
    axes[i].hist(df[feat], bins=30, edgecolor='white', color=sns.color_palette('husl', 7)[i])
    axes[i].set_title(f'{feat} ({unit})', fontsize=12)
    axes[i].set_xlabel(f'Value ({unit})')
    axes[i].set_ylabel('Frequency')
    axes[i].axvline(df[feat].mean(), color='red', linestyle='--', label=f'Mean: {df[feat].mean():.1f}')
    axes[i].legend(fontsize=8)

axes[7].axis('off')
plt.suptitle('Feature Distributions — All 7 Input Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
corr_matrix = df[features].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, vmin=-1, vmax=1,
    square=True, ax=ax
)
ax.set_title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()
print('Note: Low feature correlation → RandomForest does not require decorrelation. Good to go.')

## 5. Per-Crop Feature Boxplots (NPK)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

for ax, nutrient in zip(axes, ['N', 'P', 'K']):
    crop_order = df.groupby('label')[nutrient].median().sort_values().index
    df.boxplot(column=nutrient, by='label', ax=ax, 
               vert=False, positions=range(len(crop_order)))
    ax.set_title(f'{nutrient} (kg/ha) by Crop', fontsize=12)
    ax.set_yticklabels(crop_order, fontsize=7)
    ax.set_xlabel('kg/ha')
    plt.sca(ax)
    plt.title(f'{nutrient} Distribution per Crop')

plt.suptitle('')
plt.tight_layout()
plt.show()

## 6. pH and Rainfall Insight

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter: pH vs Temperature, colored by crop
unique_crops = df['label'].unique()
colors = sns.color_palette('tab20', len(unique_crops))
for i, crop in enumerate(unique_crops):
    mask = df['label'] == crop
    axes[0].scatter(df[mask]['ph'], df[mask]['temperature'], 
                    label=crop, alpha=0.6, s=15, color=colors[i])
axes[0].set_xlabel('pH')
axes[0].set_ylabel('Temperature (°C)')
axes[0].set_title('pH vs Temperature — by Crop')
axes[0].axvspan(6.5, 7.5, alpha=0.1, color='green', label='Ideal pH range')
axes[0].legend(fontsize=6, loc='upper right', ncol=2)

# Rainfall distribution
df_rain = df.groupby('label')['rainfall'].mean().sort_values()
df_rain.plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Mean Rainfall Requirement by Crop (mm)')
axes[1].set_xlabel('Rainfall (mm)')

plt.tight_layout()
plt.show()

## 7. EDA Summary

In [ ]:
print('=== EDA SUMMARY ===')
print(f'Total rows: {len(df)} (22 crops × 100 samples each)')
print(f'Missing values: {df.isnull().sum().sum()} — NONE')
print(f'Duplicate rows: {df.duplicated().sum()}')
print(f'Class balance: {df["label"].value_counts().std():.2f} std — PERFECTLY BALANCED')
print()
print('Feature ranges:')
for col in ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']:
    print(f'  {col:15s}: {df[col].min():.1f} – {df[col].max():.1f}  (mean={df[col].mean():.1f})')
print()
print('VERDICT: Dataset is clean, balanced, ready for RandomForest training.')
print('No imputation, no SMOTE needed. Proceed to preprocess.py → train.py')